# Qwen Architecture Inspection
**Task 1 — Generalization Specialist**

Before writing any adapter code, we need to answer five questions:
1. Are expert weights packed into one tensor (like Phi/OLMoE), or stored as separate `nn.ModuleList` modules?
2. Does Qwen1.5-MoE have **shared experts** (always-on) alongside routed experts?
3. What is the calling convention — how does the MoE block call its experts?
4. What are the exact dimensions (model_dim, ffn_dim, num_experts, top_k)?
5. Does `output_router_logits=True` work, and what shape does it return?

**Phi-3.5-MoE (known):**
- 32 layers, 16 experts/layer, top-2 routing
- Experts packed: `down_proj.shape = [16, 4096, 6400]`
- No shared experts
- LoRA dimension: `model_dim = 4096`

**OLMoE (known):**
- 16 layers, 64 experts/layer, top-8 routing
- Experts packed: `down_proj.shape = [64, 2048, 1024]`
- No shared experts
- LoRA dimension: `model_dim = 2048`

**Qwen1.5-MoE-A2.7B (unknown — this notebook finds out):**
- Expected: 24 layers, ~60 routing experts, top-4 routing
- Suspected shared expert architecture (unlike Phi/OLMoE)
- Everything else TBD

## Cell 1 — Install & Load Model

In [1]:
!pip install transformers accelerate -q

In [2]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = "Qwen/Qwen1.5-MoE-A2.7B"

tokenizer = AutoTokenizer.from_pretrained(model_name)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True,
)
model.eval()

print(f"Model class: {type(model).__name__}")
print(f"VRAM used: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/919 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 8 files:   0%|          | 0/8 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/387 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/144 [00:00<?, ?B/s]

Model class: Qwen2MoeForCausalLM
VRAM used: 28.64 GB


## Cell 2 — Smoke Test
Verify the model loads and generates correctly before touching internals.

In [3]:
prompt = "Explain in simple terms what a Mixture of Experts model is."
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

with torch.no_grad():
    output = model.generate(
        **inputs,
        max_new_tokens=100,
        temperature=0.7,
        do_sample=True,
    )

response = tokenizer.decode(output[0], skip_special_tokens=True)
print(response)

Explain in simple terms what a Mixture of Experts model is. Mixture of Experts is a machine learning technique that involves combining multiple different models to make a final prediction. Each model in the mixture is trained on a specific subset of the data, and the final prediction is calculated by combining the outputs of all the models. This approach can be useful when you have a complex problem that is difficult to solve with a single model. With a mixture of experts, you can leverage the strengths of different models to achieve better accuracy and performance.

Can you explain the concept of word


## Cell 3 — Top-Level Architecture
Print the high-level module tree to see naming conventions.

In [4]:
print("=" * 60)
print("TOP-LEVEL MODULE STRUCTURE")
print("=" * 60)
for name, module in model.named_children():
    print(f"  {name}: {type(module).__name__}")

print()
print("=" * 60)
print("DEPTH-2 MODULES")
print("=" * 60)
for name, module in model.named_modules():
    depth = name.count('.')
    if depth == 2:
        print(f"  {name}: {type(module).__name__}")

TOP-LEVEL MODULE STRUCTURE
  model: Qwen2MoeModel
  lm_head: Linear

DEPTH-2 MODULES
  model.layers.0: Qwen2MoeDecoderLayer
  model.layers.1: Qwen2MoeDecoderLayer
  model.layers.2: Qwen2MoeDecoderLayer
  model.layers.3: Qwen2MoeDecoderLayer
  model.layers.4: Qwen2MoeDecoderLayer
  model.layers.5: Qwen2MoeDecoderLayer
  model.layers.6: Qwen2MoeDecoderLayer
  model.layers.7: Qwen2MoeDecoderLayer
  model.layers.8: Qwen2MoeDecoderLayer
  model.layers.9: Qwen2MoeDecoderLayer
  model.layers.10: Qwen2MoeDecoderLayer
  model.layers.11: Qwen2MoeDecoderLayer
  model.layers.12: Qwen2MoeDecoderLayer
  model.layers.13: Qwen2MoeDecoderLayer
  model.layers.14: Qwen2MoeDecoderLayer
  model.layers.15: Qwen2MoeDecoderLayer
  model.layers.16: Qwen2MoeDecoderLayer
  model.layers.17: Qwen2MoeDecoderLayer
  model.layers.18: Qwen2MoeDecoderLayer
  model.layers.19: Qwen2MoeDecoderLayer
  model.layers.20: Qwen2MoeDecoderLayer
  model.layers.21: Qwen2MoeDecoderLayer
  model.layers.22: Qwen2MoeDecoderLayer
  mod

## Cell 4 — Find MoE Layers
Locate every module with 'expert', 'moe', or 'sparse' in its name or class.
**Watch for 'shared_expert'** — that would indicate Qwen's always-on expert architecture.

In [5]:
print("=" * 60)
print("ALL MoE-RELATED MODULES")
print("=" * 60)

moe_modules = []
for name, module in model.named_modules():
    class_name = type(module).__name__.lower()
    name_lower = name.lower()
    if any(kw in class_name or kw in name_lower
           for kw in ['expert', 'moe', 'sparse', 'router', 'gate']):
        depth = name.count('.')
        if depth <= 6:
            print(f"  [depth {depth}] {name}: {type(module).__name__}")
            moe_modules.append((name, module))

print(f"\nTotal MoE-related modules found: {len(moe_modules)}")
print()
print("KEY QUESTION: Is there a 'shared_expert' attribute?")
has_shared = any('shared_expert' in name for name, _ in moe_modules)
print(f"  → {'YES — Qwen uses shared experts (always-on)' if has_shared else 'NO — pure routing like OLMoE/Phi'}")

ALL MoE-RELATED MODULES
  [depth 0] : Qwen2MoeForCausalLM
  [depth 0] model: Qwen2MoeModel
  [depth 2] model.layers.0: Qwen2MoeDecoderLayer
  [depth 3] model.layers.0.self_attn: Qwen2MoeAttention
  [depth 3] model.layers.0.mlp: Qwen2MoeSparseMoeBlock
  [depth 4] model.layers.0.mlp.gate: Qwen2MoeTopKRouter
  [depth 4] model.layers.0.mlp.experts: Qwen2MoeExperts
  [depth 5] model.layers.0.mlp.experts.act_fn: SiLUActivation
  [depth 4] model.layers.0.mlp.shared_expert: Qwen2MoeMLP
  [depth 5] model.layers.0.mlp.shared_expert.gate_proj: Linear
  [depth 5] model.layers.0.mlp.shared_expert.up_proj: Linear
  [depth 5] model.layers.0.mlp.shared_expert.down_proj: Linear
  [depth 5] model.layers.0.mlp.shared_expert.act_fn: SiLUActivation
  [depth 4] model.layers.0.mlp.shared_expert_gate: Linear
  [depth 3] model.layers.0.input_layernorm: Qwen2MoeRMSNorm
  [depth 3] model.layers.0.post_attention_layernorm: Qwen2MoeRMSNorm
  [depth 2] model.layers.1: Qwen2MoeDecoderLayer
  [depth 3] model.layers.1

## Cell 5 — Inspect One Full Layer
Print the complete structure of layer 0, including all parameter shapes.
This reveals: packed vs separate experts, shared expert, and exact dimensions.

In [6]:
print("=" * 60)
print("FULL STRUCTURE OF LAYER 0")
print("=" * 60)

layer0 = None
for attr in ['model', 'transformer']:
    if hasattr(model, attr):
        inner = getattr(model, attr)
        for sub_attr in ['layers', 'blocks', 'h']:
            if hasattr(inner, sub_attr):
                layer0 = getattr(inner, sub_attr)[0]
                print(f"Path: model.{attr}.{sub_attr}[0]")
                break
    if layer0 is not None:
        break

if layer0 is None:
    raise RuntimeError("Could not find layer 0 — inspect model structure above")

print()
for name, module in layer0.named_modules():
    if name == '':
        continue
    depth = name.count('.')
    indent = '  ' * depth
    print(f"{indent}{name}: {type(module).__name__}")

print()
print("PARAMETERS IN LAYER 0:")
for name, param in layer0.named_parameters():
    print(f"  {name}: {param.shape} dtype={param.dtype}")

FULL STRUCTURE OF LAYER 0
Path: model.model.layers[0]

self_attn: Qwen2MoeAttention
  self_attn.q_proj: Linear
  self_attn.k_proj: Linear
  self_attn.v_proj: Linear
  self_attn.o_proj: Linear
mlp: Qwen2MoeSparseMoeBlock
  mlp.gate: Qwen2MoeTopKRouter
  mlp.experts: Qwen2MoeExperts
    mlp.experts.act_fn: SiLUActivation
  mlp.shared_expert: Qwen2MoeMLP
    mlp.shared_expert.gate_proj: Linear
    mlp.shared_expert.up_proj: Linear
    mlp.shared_expert.down_proj: Linear
    mlp.shared_expert.act_fn: SiLUActivation
  mlp.shared_expert_gate: Linear
input_layernorm: Qwen2MoeRMSNorm
post_attention_layernorm: Qwen2MoeRMSNorm

PARAMETERS IN LAYER 0:
  self_attn.q_proj.weight: torch.Size([2048, 2048]) dtype=torch.float16
  self_attn.q_proj.bias: torch.Size([2048]) dtype=torch.float16
  self_attn.k_proj.weight: torch.Size([2048, 2048]) dtype=torch.float16
  self_attn.k_proj.bias: torch.Size([2048]) dtype=torch.float16
  self_attn.v_proj.weight: torch.Size([2048, 2048]) dtype=torch.float16
  self_

## Cell 6 — CRITICAL: Packed vs Separate Experts, Shared Expert Check

**Phi/OLMoE pack all experts into a single 3D tensor:**
```
experts.down_proj.shape = [num_experts, out_f, in_f]
```

**Qwen1.5-MoE may use separate MLP modules per expert:**
```
experts = nn.ModuleList([MLP(), MLP(), ...])
```

**Qwen may also have a shared expert that always activates:**
```
shared_expert: MLP   (always fires, no gating)
shared_expert_gate: Linear(hidden_size, 1)  (optional scalar gate)
```

The packing and shared-expert answers determine how much of `HierarchicalOLMoEExperts`
can be reused and what Qwen-specific changes are required.

In [7]:
print("=" * 60)
print("PACKED vs SEPARATE EXPERTS + SHARED EXPERT CHECK")
print("=" * 60)

moe_block = None
for attr in ['mlp', 'block_sparse_moe', 'moe', 'ffn', 'feed_forward']:
    if hasattr(layer0, attr):
        moe_block = getattr(layer0, attr)
        print(f"Found MoE block at layer0.{attr}: {type(moe_block).__name__}")
        break

if moe_block is None:
    print("Could not auto-find MoE block — update path from Cell 4")
else:
    # --- Expert container ---
    expert_container = None
    for attr in ['experts', 'ffn', 'mlp']:
        if hasattr(moe_block, attr):
            expert_container = getattr(moe_block, attr)
            print(f"Expert container: .{attr}  ({type(expert_container).__name__})")
            break

    print()
    if expert_container is not None:
        if isinstance(expert_container, torch.nn.ModuleList):
            print("RESULT: SEPARATE modules (nn.ModuleList)")
            print(f"  Number of experts: {len(expert_container)}")
            print(f"  Expert[0] type: {type(expert_container[0]).__name__}")
            print(f"  Expert[0] parameters:")
            for pname, param in expert_container[0].named_parameters():
                print(f"    {pname}: {param.shape}")
        elif hasattr(expert_container, 'down_proj'):
            dp = expert_container.down_proj
            if hasattr(dp, 'shape') and len(dp.shape) == 3:
                print("RESULT: PACKED tensor (like Phi/OLMoE)")
                print(f"  down_proj.shape: {dp.shape}")
                print(f"  → [num_experts={dp.shape[0]}, out_f={dp.shape[1]}, in_f={dp.shape[2]}]")
        else:
            print("Unknown packing — all expert_container attributes:")
            for aname in dir(expert_container):
                if not aname.startswith('_'):
                    val = getattr(expert_container, aname, None)
                    if isinstance(val, (torch.Tensor, torch.nn.Module, torch.nn.ModuleList)):
                        print(f"  .{aname}: {type(val).__name__}")
                        if isinstance(val, torch.Tensor):
                            print(f"    shape: {val.shape}")

    # --- Shared expert check ---
    print()
    print("-" * 40)
    print("SHARED EXPERT CHECK:")
    for se_attr in ['shared_expert', 'shared_experts', 'shared_ffn', 'shared_mlp']:
        if hasattr(moe_block, se_attr):
            se = getattr(moe_block, se_attr)
            print(f"  Found: moe_block.{se_attr} = {type(se).__name__}")
            for pname, param in se.named_parameters():
                print(f"    {pname}: {param.shape}")
            break
    else:
        print("  No shared expert found — pure routing (like OLMoE/Phi)")

    for sge_attr in ['shared_expert_gate', 'shared_gate']:
        if hasattr(moe_block, sge_attr):
            sge = getattr(moe_block, sge_attr)
            print(f"  Found: moe_block.{sge_attr} = {type(sge).__name__}")
            if hasattr(sge, 'weight'):
                print(f"    weight.shape: {sge.weight.shape}")
            break

PACKED vs SEPARATE EXPERTS + SHARED EXPERT CHECK
Found MoE block at layer0.mlp: Qwen2MoeSparseMoeBlock
Expert container: .experts  (Qwen2MoeExperts)

RESULT: PACKED tensor (like Phi/OLMoE)
  down_proj.shape: torch.Size([60, 2048, 1408])
  → [num_experts=60, out_f=2048, in_f=1408]

----------------------------------------
SHARED EXPERT CHECK:
  Found: moe_block.shared_expert = Qwen2MoeMLP
    gate_proj.weight: torch.Size([5632, 2048])
    up_proj.weight: torch.Size([5632, 2048])
    down_proj.weight: torch.Size([2048, 5632])
  Found: moe_block.shared_expert_gate = Linear
    weight.shape: torch.Size([1, 2048])


## Cell 7 — Key Dimensions
Find `model_dim`, `ffn_dim`, number of experts (routing + shared), top-k.

**Critical:** LoRA must live in `model_dim → model_dim` space.
Using `ffn_dim` was the bug found in the Phi→OLMoE port — do not repeat it.

In [8]:
print("=" * 60)
print("KEY DIMENSIONS")
print("=" * 60)

cfg = model.config
print(f"Config class: {type(cfg).__name__}")
print()

dimension_keys = [
    'hidden_size', 'intermediate_size', 'num_hidden_layers',
    'num_attention_heads', 'num_key_value_heads',
    'num_experts', 'num_experts_per_tok', 'num_local_experts',
    'top_k', 'router_top_k', 'moe_top_k',
    'num_shared_experts', 'shared_expert_intermediate_size',
    'moe_intermediate_size', 'expert_intermediate_size',
    'ffn_dim', 'model_dim',
]

found = {}
for key in dimension_keys:
    if hasattr(cfg, key):
        val = getattr(cfg, key)
        print(f"  config.{key} = {val}")
        found[key] = val

print()
print("Other config fields (any int 10–100000):")
for key, val in vars(cfg).items():
    if key not in dimension_keys and isinstance(val, int) and 10 <= val <= 100000:
        print(f"  config.{key} = {val}")

print()
print("=" * 60)
print("INTERPRETATION:")
model_dim  = found.get('hidden_size', '???')
ffn_dim    = (found.get('moe_intermediate_size')
              or found.get('expert_intermediate_size')
              or found.get('intermediate_size', '???'))
n_experts  = found.get('num_experts') or found.get('num_local_experts', '???')
n_shared   = found.get('num_shared_experts', 0)
top_k      = (found.get('num_experts_per_tok')
              or found.get('top_k')
              or found.get('router_top_k', '???'))
print(f"  model_dim (LoRA dimension)   = {model_dim}")
print(f"  ffn_dim  (expert internal)   = {ffn_dim}")
print(f"  num routing experts per layer = {n_experts}")
print(f"  num shared experts per layer  = {n_shared}")
print(f"  top_k routing                = {top_k}")
print()
if isinstance(n_experts, int) and isinstance(top_k, int):
    baseline = top_k / n_experts * 100
    print(f"  Random Jaccard baseline: {top_k}/{n_experts} = {baseline:.1f}%")
print()
print("CRITICAL: LoRA correction must live in model_dim → model_dim space")
print(f"          i.e. lora_in = lora_out = {model_dim}")
print(f"          NOT ffn_dim ({ffn_dim}) — that was the Phi bug")

KEY DIMENSIONS
Config class: Qwen2MoeConfig

  config.hidden_size = 2048
  config.intermediate_size = 5632
  config.num_hidden_layers = 24
  config.num_attention_heads = 16
  config.num_key_value_heads = 16
  config.num_experts = 60
  config.num_experts_per_tok = 4
  config.shared_expert_intermediate_size = 5632
  config.moe_intermediate_size = 1408

Other config fields (any int 10–100000):
  config.max_position_embeddings = 8192
  config.max_window_layers = 21

INTERPRETATION:
  model_dim (LoRA dimension)   = 2048
  ffn_dim  (expert internal)   = 1408
  num routing experts per layer = 60
  num shared experts per layer  = 0
  top_k routing                = 4

  Random Jaccard baseline: 4/60 = 6.7%

CRITICAL: LoRA correction must live in model_dim → model_dim space
          i.e. lora_in = lora_out = 2048
          NOT ffn_dim (1408) — that was the Phi bug


## Cell 8 — Shared Expert Deep Dive

If Qwen uses shared experts (found in Cell 6), this cell answers:
- Is the shared expert gated by a scalar (sigmoid gate) or always fires with weight 1?
- Does the MoE block add shared expert output to routing output, or concat?
- Does routing use `output_router_logits` only for the routing experts (not shared)?

This determines whether shared experts need their own LoRA or can be left frozen.

In [9]:
import inspect

print("=" * 60)
print("MoE BLOCK forward() SOURCE")
print("=" * 60)

printed = set()
for name, module in model.named_modules():
    class_name = type(module).__name__
    if class_name in printed:
        continue
    if any(kw in class_name.lower() for kw in ['moe', 'sparse', 'mixture']):
        printed.add(class_name)
        print(f"\nClass: {class_name} (path: {name})")
        print("-" * 50)
        try:
            src = inspect.getsource(type(module).forward)
            print(src)
        except Exception as e:
            print(f"Could not get source: {e}")

MoE BLOCK forward() SOURCE

Class: Qwen2MoeForCausalLM (path: )
--------------------------------------------------
    @can_return_tuple
    @auto_docstring
    def forward(
        self,
        input_ids: torch.LongTensor | None = None,
        attention_mask: torch.Tensor | None = None,
        position_ids: torch.LongTensor | None = None,
        past_key_values: Cache | None = None,
        inputs_embeds: torch.FloatTensor | None = None,
        labels: torch.LongTensor | None = None,
        use_cache: bool | None = None,
        output_router_logits: bool | None = None,
        cache_position: torch.LongTensor | None = None,
        logits_to_keep: int | torch.Tensor = 0,
        **kwargs: Unpack[TransformersKwargs],
    ) -> MoeCausalLMOutputWithPast:
        r"""
        labels (`torch.LongTensor` of shape `(batch_size, sequence_length)`, *optional*):
            Labels for computing the masked language modeling loss. Indices should either be in `[0, ...,
            config.vo

## Cell 9 — Router Structure
Find the router / gate module and understand its forward pass.
We need: softmax+topk convention, output signature, weight shape.

In [10]:
print("=" * 60)
print("ROUTER / GATE STRUCTURE")
print("=" * 60)

printed_routers = set()
for name, module in model.named_modules():
    class_name = type(module).__name__.lower()
    depth = name.count('.')
    if depth > 6:
        continue
    if ('router' in class_name or
        ('gate' in name.lower() and 'layer' in name.lower() and 'shared' not in name.lower())):
        key = type(module).__name__
        if key in printed_routers:
            continue
        printed_routers.add(key)
        print(f"Found: {name}")
        print(f"  Type: {type(module).__name__}")
        for pname, param in module.named_parameters():
            print(f"  {pname}: {param.shape}")
        try:
            src = inspect.getsource(type(module).forward)
            print("  forward() source:")
            for line in src.split('\n')[:35]:
                print(f"    {line}")
        except Exception:
            pass
        print()

ROUTER / GATE STRUCTURE
Found: model.layers.0.mlp.gate
  Type: Qwen2MoeTopKRouter
  weight: torch.Size([60, 2048])
  forward() source:
        def forward(self, hidden_states):
            hidden_states = hidden_states.reshape(-1, self.hidden_dim)
            router_logits = F.linear(hidden_states, self.weight)  # (seq_len, num_experts)
            router_logits = torch.nn.functional.softmax(router_logits, dtype=torch.float, dim=-1)
            router_top_value, router_indices = torch.topk(router_logits, self.top_k, dim=-1)  # (seq_len, top_k)
            if self.norm_topk_prob:
                router_top_value /= router_top_value.sum(dim=-1, keepdim=True)
            router_top_value = router_top_value.to(router_logits.dtype)
            router_scores = router_top_value
            return router_logits, router_scores, router_indices
    



## Cell 10 — Test `output_router_logits`

The Jaccard diagnostic and `ConflictSaturationMonitor` both need per-layer routing
decisions. Test whether `output_router_logits=True` works and what shape returns.

**Note for Qwen:** If there are shared experts, `router_logits` should cover only
the routing experts (not shared) — verify this below.

In [11]:
print("=" * 60)
print("output_router_logits TEST")
print("=" * 60)

test_input = tokenizer("The patient presented with fever and chills.",
                        return_tensors="pt").to(model.device)

with torch.no_grad():
    try:
        out = model(**test_input, output_router_logits=True, return_dict=True)
        print("✅ output_router_logits=True is supported")
        print()
        if hasattr(out, 'router_logits') and out.router_logits is not None:
            logits_list = out.router_logits
            print(f"router_logits type: {type(logits_list).__name__}")
            print(f"Number of elements (should = num_layers): {len(logits_list)}")
            print()
            for i, rl in enumerate(logits_list[:4]):
                if rl is not None:
                    print(f"  Layer {i}: shape={rl.shape}, dtype={rl.dtype}")
                else:
                    print(f"  Layer {i}: None (non-MoE layer?)")
            print("  ...")
            print()
            # Find first non-None logits
            rl0 = next((r for r in logits_list if r is not None), None)
            if rl0 is not None:
                print("INTERPRETATION of first non-None router_logits:")
                if rl0.dim() == 2:
                    T, E = rl0.shape
                    print(f"  Shape: [{T} tokens, {E} experts]")
                    print(f"  → (batch*seq, n_routing_experts) — same as OLMoE")
                elif rl0.dim() == 3:
                    B, S, E = rl0.shape
                    print(f"  Shape: [{B} batch, {S} seq, {E} experts]")
                    print(f"  → needs .view(-1, {E}) before topk")
        else:
            print("❌ router_logits is None or missing")
            print(f"   Output keys: {list(out.keys()) if hasattr(out, 'keys') else dir(out)}")
    except Exception as e:
        print(f"❌ output_router_logits=True raised: {e}")
        print("   Will use forward hook approach instead (see Cell 11)")

output_router_logits TEST
✅ output_router_logits=True is supported

router_logits type: tuple
Number of elements (should = num_layers): 24

  Layer 0: shape=torch.Size([9, 60]), dtype=torch.float32
  Layer 1: shape=torch.Size([9, 60]), dtype=torch.float32
  Layer 2: shape=torch.Size([9, 60]), dtype=torch.float32
  Layer 3: shape=torch.Size([9, 60]), dtype=torch.float32
  ...

INTERPRETATION of first non-None router_logits:
  Shape: [9 tokens, 60 experts]
  → (batch*seq, n_routing_experts) — same as OLMoE


## Cell 11 — Forward Hook Fallback
If `output_router_logits` is not supported, intercept routing with forward hooks.
Also useful to inspect shared-expert gating separately from routing experts.

In [12]:
print("=" * 60)
print("FORWARD HOOK FALLBACK")
print("=" * 60)

captured_router_outputs = {}
hooks = []

for name, module in model.named_modules():
    class_name = type(module).__name__.lower()
    if ('router' in class_name or
        ('gate' in name.lower() and 'layer' in name.lower() and 'shared' not in name.lower())):
        def make_hook(n):
            def hook(mod, inp, out):
                captured_router_outputs[n] = out
            return hook
        h = module.register_forward_hook(make_hook(name))
        hooks.append(h)

test_input = tokenizer("The patient presented with fever.",
                        return_tensors="pt").to(model.device)

with torch.no_grad():
    _ = model(**test_input)

for h in hooks:
    h.remove()

print(f"Captured outputs from {len(captured_router_outputs)} router modules")
print()
for name, out in list(captured_router_outputs.items())[:3]:
    print(f"Router: {name}")
    if isinstance(out, tuple):
        print(f"  Output is a tuple of {len(out)} elements:")
        for i, elem in enumerate(out):
            if isinstance(elem, torch.Tensor):
                print(f"    [{i}]: shape={elem.shape}, dtype={elem.dtype}")
    elif isinstance(out, torch.Tensor):
        print(f"  Output tensor: shape={out.shape}, dtype={out.dtype}")
    print()

FORWARD HOOK FALLBACK
Captured outputs from 24 router modules

Router: model.layers.0.mlp.gate
  Output is a tuple of 3 elements:
    [0]: shape=torch.Size([6, 60]), dtype=torch.float32
    [1]: shape=torch.Size([6, 4]), dtype=torch.float32
    [2]: shape=torch.Size([6, 4]), dtype=torch.int64

Router: model.layers.1.mlp.gate
  Output is a tuple of 3 elements:
    [0]: shape=torch.Size([6, 60]), dtype=torch.float32
    [1]: shape=torch.Size([6, 4]), dtype=torch.float32
    [2]: shape=torch.Size([6, 4]), dtype=torch.int64

Router: model.layers.2.mlp.gate
  Output is a tuple of 3 elements:
    [0]: shape=torch.Size([6, 60]), dtype=torch.float32
    [1]: shape=torch.Size([6, 4]), dtype=torch.float32
    [2]: shape=torch.Size([6, 4]), dtype=torch.int64



## Cell 12 — Verify Zero-Output LoRA Patch

Core guarantee: a LoRA adapter with `B=0` produces zero output.
Patching must leave the model's behavior unchanged at initialization.

This is architecture-agnostic — same test as OLMoE.

In [13]:
import torch.nn as nn

print("=" * 60)
print("ZERO-OUTPUT LoRA PATCH VERIFICATION")
print("=" * 60)

model_dim = model.config.hidden_size
rank = 16

lora_A = nn.Parameter(torch.randn(rank, model_dim, dtype=torch.float16) * 0.01)
lora_B = nn.Parameter(torch.zeros(model_dim, rank, dtype=torch.float16))  # B=0

dummy_input = torch.randn(5, model_dim, dtype=torch.float16)
lora_out = (dummy_input @ lora_A.T) @ lora_B.T

print(f"LoRA output with B=0:")
print(f"  max absolute value:  {lora_out.abs().max().item():.8f}")
print(f"  mean absolute value: {lora_out.abs().mean().item():.8f}")

if lora_out.abs().max().item() < 1e-6:
    print("✅ Zero-output guarantee confirmed — B=0 produces exactly zero")
else:
    print("❌ Unexpected non-zero output — check initialization")

# Also verify a forward pass gives same output before and after a null patch
print()
print("Verifying model output unchanged with null LoRA correction...")
moe_layer = None
for attr in ['mlp', 'block_sparse_moe', 'moe', 'ffn', 'feed_forward']:
    if hasattr(model.model.layers[0], attr):
        moe_layer = getattr(model.model.layers[0], attr)
        print(f"Using layer0.{attr} for hook")
        break

captured_out = {}
if moe_layer is not None:
    test_in = tokenizer("Hello world", return_tensors="pt").to(model.device)
    def capture(mod, inp, out):
        if isinstance(out, tuple):
            captured_out['out'] = out[0].detach().clone()
        elif isinstance(out, torch.Tensor):
            captured_out['out'] = out.detach().clone()
    h = moe_layer.register_forward_hook(capture)
    with torch.no_grad():
        _ = model(**test_in)
    h.remove()
    print(f"  Captured MoE block output shape: {captured_out.get('out', torch.tensor(0)).shape}")
    print("  (This is the reference output — patching with B=0 should give identical result)")

ZERO-OUTPUT LoRA PATCH VERIFICATION
LoRA output with B=0:
  max absolute value:  0.00000000
  mean absolute value: 0.00000000
✅ Zero-output guarantee confirmed — B=0 produces exactly zero

Verifying model output unchanged with null LoRA correction...
Using layer0.mlp for hook
  Captured MoE block output shape: torch.Size([1, 2, 2048])
  (This is the reference output — patching with B=0 should give identical result)


## Cell 13 — Accelerate Hook Check

When Phi was patched, the accelerate `AlignDevicesHook` had to be manually
transferred from the original expert block to the wrapper. Check if Qwen needs this.

In [14]:
print("=" * 60)
print("ACCELERATE HOOK CHECK")
print("=" * 60)

hook_found = False
for name, module in model.named_modules():
    if hasattr(module, '_hf_hook'):
        print(f"Found _hf_hook on: {name}")
        print(f"  Hook type: {type(module._hf_hook).__name__}")
        hook_found = True

if not hook_found:
    print("No _hf_hook found on any module.")
    print("→ Single-GPU case — no hook transfer needed when patching.")
    print("→ On multi-GPU (device_map='auto' with 2+ GPUs), re-run this")
    print("  check and look for AlignDevicesHook.")
else:
    print()
    print("ACTION REQUIRED when patching:")
    print("  from accelerate.hooks import remove_hook_from_module, add_hook_to_module")
    print("  hook = original_experts._hf_hook")
    print("  remove_hook_from_module(original_experts)")
    print("  add_hook_to_module(your_wrapper, hook)")

ACCELERATE HOOK CHECK
No _hf_hook found on any module.
→ Single-GPU case — no hook transfer needed when patching.
→ On multi-GPU (device_map='auto' with 2+ GPUs), re-run this
  check and look for AlignDevicesHook.


## Cell 14 — Expert Forward Convention Deep Dive

Exactly how does the MoE block dispatch tokens to experts?
We need to know what arguments the expert block receives and returns.

If experts are **separate modules** (ModuleList), the block likely loops:
```python
for i, expert in enumerate(self.experts):
    mask = (selected_experts == i)
    output[mask] = expert(hidden_states[mask])
```

If experts are **packed** (like OLMoE), the block calls:
```python
output = self.experts(hidden_states, top_k_ids, top_k_weights)
```

The wrapper we write must match the correct convention.

In [15]:
import inspect

print("=" * 60)
print("EXPERT MLP forward() SOURCE")
print("=" * 60)

# Inspect the expert MLP class (could be Qwen2MoeMLP or similar)
expert_container = None
for attr in ['mlp', 'block_sparse_moe', 'moe', 'ffn', 'feed_forward']:
    if hasattr(layer0, attr):
        moe_block = getattr(layer0, attr)
        for ea in ['experts', 'ffn', 'mlp']:
            if hasattr(moe_block, ea):
                expert_container = getattr(moe_block, ea)
                break
        break

printed_types = set()
if isinstance(expert_container, torch.nn.ModuleList) and len(expert_container) > 0:
    expert_cls = type(expert_container[0])
    if expert_cls not in printed_types:
        printed_types.add(expert_cls)
        print(f"Expert class: {expert_cls.__name__}")
        try:
            src = inspect.getsource(expert_cls.forward)
            print(src)
        except Exception as e:
            print(f"Could not get source: {e}")
elif expert_container is not None:
    expert_cls = type(expert_container)
    print(f"Expert container class: {expert_cls.__name__}")
    try:
        src = inspect.getsource(expert_cls.forward)
        print(src)
    except Exception as e:
        print(f"Could not get source: {e}")

EXPERT MLP forward() SOURCE
Expert container class: Qwen2MoeExperts
    def forward(
        self,
        hidden_states: torch.Tensor,
        top_k_index: torch.Tensor,
        top_k_weights: torch.Tensor,
    ) -> torch.Tensor:
        final_hidden_states = torch.zeros_like(hidden_states)
        with torch.no_grad():
            expert_mask = torch.nn.functional.one_hot(top_k_index, num_classes=self.num_experts)
            expert_mask = expert_mask.permute(2, 1, 0)
            expert_hit = torch.greater(expert_mask.sum(dim=(-1, -2)), 0).nonzero()

        for expert_idx in expert_hit:
            expert_idx = expert_idx[0]
            if expert_idx == self.num_experts:
                continue
            top_k_pos, token_idx = torch.where(expert_mask[expert_idx])
            current_state = hidden_states[token_idx]
            gate, up = nn.functional.linear(current_state, self.gate_up_proj[expert_idx]).chunk(2, dim=-1)
            current_hidden_states = self.act_fn(gate) * up
 

## Cell 15 — Summary and Comparison Table

Collect all findings and compare with Phi-3.5-MoE and OLMoE.
This table determines exactly what needs to change in `HierarchicalOLMoEExperts`
to produce `HierarchicalQwenExperts`.

In [16]:
cfg_obj = model.config

model_dim  = getattr(cfg_obj, 'hidden_size', '???')
ffn_dim    = (getattr(cfg_obj, 'moe_intermediate_size', None)
              or getattr(cfg_obj, 'expert_intermediate_size', None)
              or getattr(cfg_obj, 'intermediate_size', '???'))
n_layers   = getattr(cfg_obj, 'num_hidden_layers', '???')
n_experts  = getattr(cfg_obj, 'num_experts', None) or getattr(cfg_obj, 'num_local_experts', '???')
n_shared   = getattr(cfg_obj, 'num_shared_experts', 0)
top_k      = (getattr(cfg_obj, 'num_experts_per_tok', None)
              or getattr(cfg_obj, 'top_k', None)
              or getattr(cfg_obj, 'router_top_k', '???'))

if isinstance(n_experts, int) and isinstance(top_k, int):
    random_j = f"{top_k}/{n_experts}={top_k/n_experts*100:.1f}%"
else:
    random_j = '???'

print("=" * 72)
print(f"{'PROPERTY':<35} {'PHI-3.5':>10} {'OLMoE':>10} {'Qwen1.5':>10}")
print("=" * 72)
print(f"{'Num layers':<35} {'32':>10} {'16':>10} {str(n_layers):>10}")
print(f"{'Routing experts per layer':<35} {'16':>10} {'64':>10} {str(n_experts):>10}")
print(f"{'Shared experts per layer':<35} {'0':>10} {'0':>10} {str(n_shared):>10}")
print(f"{'Top-k routing':<35} {'2':>10} {'8':>10} {str(top_k):>10}")
print(f"{'model_dim (LoRA dimension)':<35} {'4096':>10} {'2048':>10} {str(model_dim):>10}")
print(f"{'ffn_dim (expert internal)':<35} {'6400':>10} {'1024':>10} {str(ffn_dim):>10}")
print(f"{'Random Jaccard baseline':<35} {'12.5%':>10} {'12.5%':>10} {random_j:>10}")
print("=" * 72)

print()
print("FILL IN MANUALLY FROM CELLS ABOVE:")
print("  Experts packed or separate modules?    → [ ]")
print("  Shared expert architecture?            → [ ]")
print("  Shared expert gate type?               → [ ]")
print("  Calling convention for experts?        → [ ]")
print("  output_router_logits=True supported?   → [ ]")
print("  router_logits shape: (T, E) or (B,S,E)?→ [ ]")
print("  Accelerate hook present?               → [ ]")
print()
print("WHAT NEEDS TO CHANGE FROM HierarchicalOLMoEExperts:")
print("  [ ] Rename class to HierarchicalQwenExperts")
print("  [ ] Handle ModuleList dispatch (if separate experts, not packed)")
print("  [ ] Freeze shared expert OR add base LoRA to it separately")
print("  [ ] Update lora_dim = model_dim = hidden_size")
print("  [ ] Update random Jaccard baseline in diagnostics")
print("  [ ] Confirm hook transfer approach")

PROPERTY                               PHI-3.5      OLMoE    Qwen1.5
Num layers                                  32         16         24
Routing experts per layer                   16         64         60
Shared experts per layer                     0          0          0
Top-k routing                                2          8          4
model_dim (LoRA dimension)                4096       2048       2048
ffn_dim (expert internal)                 6400       1024       1408
Random Jaccard baseline                  12.5%      12.5%  4/60=6.7%

FILL IN MANUALLY FROM CELLS ABOVE:
  Experts packed or separate modules?    → [ ]
  Shared expert architecture?            → [ ]
  Shared expert gate type?               → [ ]
  Calling convention for experts?        → [ ]
  output_router_logits=True supported?   → [ ]
  router_logits shape: (T, E) or (B,S,E)?→ [ ]
  Accelerate hook present?               → [ ]

WHAT NEEDS TO CHANGE FROM HierarchicalOLMoEExperts:
  [ ] Rename class to Hierarch

## Cell 16 — Save Config for Next Notebooks

In [17]:
import json

cfg_obj = model.config
model_dim = getattr(cfg_obj, 'hidden_size', 2048)
ffn_dim   = (getattr(cfg_obj, 'moe_intermediate_size', None)
             or getattr(cfg_obj, 'expert_intermediate_size', None)
             or getattr(cfg_obj, 'intermediate_size', 1408))
n_layers  = getattr(cfg_obj, 'num_hidden_layers', 24)
n_experts = getattr(cfg_obj, 'num_experts', None) or getattr(cfg_obj, 'num_local_experts', 60)
n_shared  = getattr(cfg_obj, 'num_shared_experts', 4)
top_k     = (getattr(cfg_obj, 'num_experts_per_tok', None)
             or getattr(cfg_obj, 'top_k', None)
             or getattr(cfg_obj, 'router_top_k', 4))

arch_config = {
    "model_name":         "Qwen/Qwen1.5-MoE-A2.7B",
    "model_dim":          model_dim,
    "ffn_dim":            ffn_dim,
    "num_layers":         n_layers,
    "num_routing_experts": n_experts,
    "num_shared_experts": n_shared,
    "top_k":              top_k,
    # Fill in these manually after running the cells above:
    "experts_packed":     None,   # True if packed tensor, False if ModuleList
    "has_shared_expert":  None,   # True/False
    "shared_expert_gated": None,  # True if sigmoid gate, False if always weight=1
    "router_logits_shape": None,  # "T_E" or "B_S_E"
    "accelerate_hook":    None,   # True/False
}

with open("qwen_arch_config.json", "w") as f:
    json.dump(arch_config, f, indent=2)

print("Architecture config saved to qwen_arch_config.json")
print()
print(json.dumps(arch_config, indent=2))
print()
print("NOTE: Fill in None fields manually based on cell outputs above,")
print("      then save again before running the Jaccard diagnostic notebook.")

Architecture config saved to qwen_arch_config.json

{
  "model_name": "Qwen/Qwen1.5-MoE-A2.7B",
  "model_dim": 2048,
  "ffn_dim": 1408,
  "num_layers": 24,
  "num_routing_experts": 60,
  "num_shared_experts": 4,
  "top_k": 4,
  "experts_packed": null,
  "has_shared_expert": null,
  "shared_expert_gated": null,
  "router_logits_shape": null,
  "accelerate_hook": null
}

NOTE: Fill in None fields manually based on cell outputs above,
      then save again before running the Jaccard diagnostic notebook.
